### exp_avg（一阶矩）和 exp_avg_sq（二阶矩）
- 它们通过指数移动平均来估计梯度的均值和方差，实现自适应学习率调整

这两个变量直接决定了参数更新的方向和步长。

| 变量名 | 数学符号 | 物理含义 | 作用 | 对应统计量 |
| :--- | :---: | :--- | :--- | :--- |
| `exp_avg` |  $ m_t $  | 一阶矩估计 (First Moment) | 决定更新方向。平滑梯度噪声，提供“惯性”，加速收敛。 | 均值 (Mean) / 动量 |
| `exp_avg_sq` |  $ v_t $  | 二阶矩估计 (Second Moment) | 决定更新步长。感知梯度变化幅度，<b>对频繁变化的参数减小步长，对稀疏参数增大步长。</b> | 未中心化方差 (Uncentered Variance) |



- avg = Average (平均值) --> 一阶矩。
- sq = Square (平方) --> 平方的平均 --> 二阶矩。
- exp = Exponential Moving Average (指数移动平均) --> 表示它们是历史信息的加权累积，而非简单的算术平均。

### 总结
- exp_avg = 导航仪。利用历史方向信息，让优化过程更平稳、更快（动量）。
- exp_avg_sq = 油门/刹车。利用历史波动信息，自动调节每个参数的学习步长（自适应）。
- 两者结合，使得 Adam 成为目前最通用、最鲁棒的优化器之一，但也带来了显著的显存开销（需存储两份与参数同大的状态）。

## 数学原理详解
- $ g_t $是当前计算出的梯度。
- $ β_1 , β_2 $ 是衰减率（超参数，通常$β_1=0.9 , β_2=0.999 $ )

### 🟢 exp_avg ( $ m_t $)：一阶矩 (动量)

#### 公式
$ m_t = β_1 * m_{t-1} + (1 - β_1)*g_t$

1. 指数移动平均 (EMA)：它不是简单的平均值，而是给最近的梯度更高的权重，久远的梯度权重呈指数衰减。
2. 物理意义（动量）：
- 如果梯度方向一直一致（如下坡）， $ m_t $ 会越来越大，加速下降。
- 如果梯度方向震荡（如在峡谷两侧反复横跳），正负抵消， $ m_t $会变小，抑制震荡。
3. 解决了 SGD 在平坦区域收敛慢、在震荡区域不稳定的问题

### exp_avg_sq ( $ v_t $)：二阶矩 (自适应学习率)
#### 公式
$ v_t = β_2 * v_{t-1} + (1-β_2) * g_t^2 $

(注意：这里是对梯度 $ g_t $ 先平方，再求平均)

1. 梯度的平方平均：它记录了梯度幅度的历史信息。
2. 物理意义（方差/尺度）：
   - 大梯度：如果某个参数过去的梯度很大且波动剧烈，$v_t$ 会很大
   - 小梯度：如果某个参数过去的梯度很小或很稳定，$v_t$ 会很小
   
3. 作用（自适应）：
   - 在更新公式中，学习率会被除以 $\sqrt{v_t}$
   - $v_t$大 --> 分母大 -->有效学习率变小（防止步子太大迈过去）
   - $v_t$小 --> 分母小 -->有效学习率变大 （鼓励在平台快速探索）
   
###  它们在参数更新中如何协作？
以 Adam 为例，最终的参数更新步骤如下：
#### 偏差修正 (Bias Correction)：
由于$ m_0 , v_0 初始化为0，初期估计会偏向0，需要修正 $

$\hat{m_t} = \frac{m_t}{1-β^t_1}$

$\hat{v_t} = \frac{v_t}{1-β^t_2}$

#### 计算更新量：

$θ_{t+1} = θ_t - \alpha * \frac{\hat{m_t}}{\sqrt{\hat{v_t} }+ \epsilon}$

*    $ \alpha $ ：全局学习率。
*    $ \epsilon $ ：极小值，防止除以零。

#### 协作逻辑
- 分子($\hat{m_t}$) 告诉我们要往哪个方向走（基于动量）
- 分母($\sqrt{\hat{v_t}}$) 告诉我们要走多大步子（基于历史波动）

## 代码实现视角 (PyTorch)
- 在 PyTorch 的 torch.optim.Adam 源码逻辑中，这两个变量存储在 optimizer.state[param] 字典里：

In [ ]:
# 伪代码逻辑展示
for group in optimizer.param_groups:
    for p in group['params']:
        if p.grad is None:
            continue
        
        state = optimizer.state[p]
        
        # 初始化状态
        if len(state) == 0:
            state['step'] = 0
            # exp_avg: 一阶矩 (动量)
            state['exp_avg'] = torch.zeros_like(p.data)
            # exp_avg_sq: 二阶矩 (平方梯度平均)
            state['exp_avg_sq'] = torch.zeros_like(p.data)
            
        state['step'] += 1
        
        beta1, beta2 = group['betas']
        
        # 1. 更新一阶矩 (exp_avg)
        state['exp_avg'].mul_(beta1).add_(p.grad, alpha=1 - beta1)
        
        # 2. 更新二阶矩 (exp_avg_sq)
        # 注意：这里是 gradient 的平方
        state['exp_avg_sq'].mul_(beta2).addcmul_(p.grad, p.grad, value=1 - beta2)
        
        # 3. 偏差修正 & 更新参数
        # ... (省略偏差修正和最终更新公式)

### 显存占用分析

对于每个参数（比如一个权重矩阵），我们需要额外存储两个同形状的浮点张量：
- exp_avg (FP32)
- exp_avg_sq (FP32)
- 加上参数本身的主副本 (Master Weight, FP32)。

结论：使用 Adam 优化器时，显存占用通常是模型参数量的 3倍~4倍 (参数 + 动量 + 方差 + 梯度)。这也是为什么在大模型训练中需要使用 ZeRO-1 来分片 exp_avg 和 exp_avg_sq 的原因。

### ❓ 在混合精度训练 (AMP) 中要注意什么？

- exp_avg 和 exp_avg_sq <b>必须保持在 FP32 精度。</b>
- <b>如果用 FP16 存储它们，exp_avg_sq 极易发生下溢 (Underflow) 变成 0（因为梯度平方后很小），导致分母为 0 或学习率异常大，训练崩溃。</b>
- 这也是为什么优化器状态（Optimizer States）占显存的主要原因。